# Okta + Vault: Agentic IAM y OBO

Instalación reproducible con Terraform y pruebas OAuth desde comandos shell. Ejecutar con la carpeta `okta/` como directorio de trabajo. El material queda aparcado hasta disponer del SKU NHI. No se muestran claves ni tokens. Los helpers Python usan HTTPS y guardan las credenciales temporales en Vault.

**Estado al 13-10-2026:** 15 recursos desplegados. Login humano con FastPass y PKCE completado; JWT verificado y alias humano creado. Client Credentials y Token Exchange OBO con subject token válido devuelven que falta el SKU NHI Authentication Tokens. La prueba completa de acceso del agente a Vault queda pendiente de esa capacidad. Este notebook no contiene outputs simulados.

Documentación: [Okta](https://developer.hashicorp.com/vault/ai/iam/providers/okta), [agente](https://developer.hashicorp.com/vault/ai/iam/setup-agent), [OBO](https://developer.hashicorp.com/vault/ai/iam/setup-obo), [RAR](https://developer.hashicorp.com/vault/ai/iam/make-rar-optional). Ver [README](README.md) para el detalle del despliegue y [bootstrap](docs/bootstrap-okta.md) para las capturas iniciales.

## 1. Preparar herramientas

Requisitos: Terraform 1.15+, Python 3.11+ recomendado, Vault disponible y `/tmp/vault/config.env` con `VAULT_ADDR`, `VAULT_CACERT`, `VAULT_TOKEN`. El bootstrap administrativo debe existir en `secrets/okta`. El wrapper no imprime esos valores.

La credencial de bootstrap administra aplicaciones y servidores de autorización de esta organización. Las identidades de la demo reciben únicamente permisos de lectura sobre rutas sintéticas concretas.

In [ ]:
%%bash
set -euo pipefail
test -f scripts/terraform_okta.py
test -r /tmp/vault/config.env
terraform version
python3 -m venv .venv
.venv/bin/python -m pip install -r requirements-agentic.txt


## 2. Inicializar y validar Terraform

Se administran los recursos de Okta y Vault en `terraform/okta`. Estado y planes permanecen fuera de Google Drive, en `~/.local/share/okta-vault-agentic/`. Conservar este directorio y el lockfile. No ejecutar Terraform directamente sin el wrapper, pues podría utilizar otro backend o credenciales.

La aplicación humana usa Authorization Code + PKCE y Refresh Token. El agente usa Client Credentials y Token Exchange. La política del servidor `default` se limita a ambas aplicaciones. `vault.read` se configura como scope por defecto, pero los permisos efectivos los evalúa Vault.

In [ ]:
%%bash
set -euo pipefail
python3 scripts/terraform_okta.py init -input=false
terraform -chdir=terraform/okta fmt -check
python3 scripts/terraform_okta.py validate -no-color


## 3. Revisar el plan y aplicar

La primera celda guarda el plan. Revisar la salida antes de ejecutar la segunda. El plan no incluye los client secrets (`omit_secret = true`). En una instalación ya aplicada puede mostrar cero cambios.

In [ ]:
%%bash
set -euo pipefail
python3 scripts/terraform_okta.py plan -input=false -no-color   -out="$HOME/.local/share/okta-vault-agentic/notebook.tfplan"


In [ ]:
%%bash
set -euo pipefail
python3 scripts/terraform_okta.py apply -input=false -no-color   "$HOME/.local/share/okta-vault-agentic/notebook.tfplan"


## 4. Guardar los client secrets desde Chrome

Después de la creación, abrir en Okta las aplicaciones **Vault Agentic IAM Agent** y **Vault Agentic IAM Human**. En **General → Client secrets**, copiar el valor real. En Vault, crear o actualizar KV v2 `secrets/okta-agentic` desde el editor JSON.

Campos: `client_id`, `client_secret`, `human_client_id`, `human_client_secret`, `issuer`, `jwks_uri`, `audience`, `scope`, `redirect_uri`. Los campos públicos están documentados en el README. No pegar los secretos en una celda de Jupyter.

Este paso ya se completó en la instalación actual. Repetirlo únicamente si se recrean o rotan los clientes.

![Aplicaciones](docs/screenshots/11-okta-terraform-apps.jpg)
![Vault, sin valores](docs/screenshots/12-vault-agentic-credentials.jpg)

## 5. Verificar configuración persistida

El perfil valida RS256, issuer, audience y `sub`. El alias del agente incluye `issuer` y el Client ID como `external_id`. El perfil exige RAR; la excepción se limita al registro de este agente. Su ceiling permite leer exclusivamente `secrets/data/agentic-demo/shared`.

La entidad humana existe con lectura de `shared` y `human-only`. Su alias se añade tras validar un token humano real. Esta comprobación de configuración **no prueba** que Okta emita tokens ni que OBO funcione.

In [ ]:
%%bash
set -euo pipefail
.venv/bin/python scripts/verify_configuration.py


## 6. Probar el agente autónomo

La prueba solicita un token con `client_credentials` y `vault.read`, verifica criptográficamente el JWT y comprueba cuatro operaciones en Vault. Resultado esperado: lectura de shared=200; escritura=403; lectura de human-only=403; lectura del bootstrap=403.

**Bloqueo observado:** esta organización devuelve `invalid_grant` y falta de **NHI Authentication Tokens SKU**. Solicitar su activación a Okta o utilizar una organización que lo incluya antes de esperar una prueba satisfactoria. No interpretar que seleccionar el grant en la UI habilita la licencia.

In [ ]:
%%bash
set -euo pipefail
.venv/bin/python scripts/agentic_demo.py machine


## 7. Login humano con Authorization Code + PKCE

FastPass ya está activado: **Security → Authenticators → Okta Verify → Actions → Edit → Okta FastPass (All platforms)**.

Para el flujo desde Chrome en el Mac: portal de usuario → nombre → Settings → Security methods → Okta Verify → Set up another. Registrar Okta Verify en el Mac y habilitar la verificación del dispositivo. El TOTP del teléfono no cumple por sí solo el requisito resistente al phishing.

![FastPass habilitado](docs/screenshots/15-okta-fastpass-enabled.jpg)

La siguiente celda imprime una URL y espera hasta 10 minutos. Abrirla en Chrome y completar MFA. El callback escucha solo en 127.0.0.1:9876. Si expira, volver a ejecutar la celda; no reutilizar la URL anterior. Los tokens se guardan en Vault, no en los outputs. El `sub` verificado se guarda en un archivo local protegido para Terraform.

In [ ]:
%%bash
set -euo pipefail
.venv/bin/python scripts/agentic_demo.py login


## 8. Crear el alias humano

Ejecutar solo después del login correcto. El wrapper carga automáticamente `~/.local/share/okta-vault-agentic/human.tfvars.json`. Terraform añade el alias asociado al humano y al issuer de Okta. No se modifica el token ni se inventa su identidad.

In [ ]:
%%bash
set -euo pipefail
test -s "$HOME/.local/share/okta-vault-agentic/human.tfvars.json"
python3 scripts/terraform_okta.py plan -input=false -no-color   -out="$HOME/.local/share/okta-vault-agentic/human.tfplan"


In [ ]:
%%bash
set -euo pipefail
python3 scripts/terraform_okta.py apply -input=false -no-color   "$HOME/.local/share/okta-vault-agentic/human.tfplan"


## 9. Intercambio OBO y comprobación de límites

El agente se autentica con su propio cliente y presenta el access token humano como `subject_token` al endpoint de Okta. Se requiere `grant_type=urn:ietf:params:oauth:grant-type:token-exchange`, `subject_token_type=urn:ietf:params:oauth:token-type:access_token`, scope `vault.read` y audience `api://default`.

El helper valida firma, caducidad y audience, exige que `sub` sea el humano y `act.sub` sea el agente, y luego prueba Vault. Un JWT sin actor correcto no se acepta como evidencia OBO. Los errores del endpoint se muestran sin imprimir tokens.

**Resultado real:** el intercambio se probó con un token humano válido después de configurar FastPass. Okta devolvió HTTP 400 `invalid_grant`: falta NHI Authentication Tokens SKU. Habilitarlo antes de repetir esta celda. No se obtuvo un token delegado ni se ejecutaron todavía las pruebas de permisos con él.

In [ ]:
%%bash
set -euo pipefail
.venv/bin/python scripts/agentic_demo.py obo


## 10. Repetir las comprobaciones

Requiere ambos tokens sin caducar. Si caducan, repetir las celdas de obtención. `verify_configuration.py` genera un informe independiente en `docs/verification.json`.

No se usa `vault kv get` con los JWT externos: las pruebas llaman a rutas KV v2 explícitas. No se solicita acceso a ningún secreto real fuera de la prueba de denegación del bootstrap.

In [ ]:
%%bash
set -euo pipefail
.venv/bin/python scripts/agentic_demo.py verify
.venv/bin/python scripts/verify_configuration.py
python3 scripts/terraform_okta.py plan -input=false -no-color -detailed-exitcode


## Referencias y conservación

- [OBO de Okta, con claim act](https://developer.okta.com/docs/guides/set-up-token-exchange/main/)
- [Registro de agentes en Terraform](https://github.com/hashicorp/terraform-provider-vault/blob/v5.12.0/website/docs/r/agent_registration.html.md)
- [Perfil OAuth en Terraform](https://github.com/hashicorp/terraform-provider-vault/blob/v5.12.0/website/docs/r/oauth_resource_server_config_profile.html.md)

El estado debe mantenerse para gestionar el ciclo de vida. Los client secrets residen en `secrets/okta-agentic`; el bootstrap en `secrets/okta`; los tokens obtenidos en `secrets/okta-agentic-session`. KV conserva versiones aunque los tokens OAuth hayan expirado. No se incluye una celda de borrado automático de infraestructura o credenciales.

## Comprobación adicional del trial de Okta

Comprobado el 13 de octubre de 2026 en Chrome: `trial-2604851.okta.com` tiene **23 días restantes**. Se creó el cliente **Vault NHI Trial Probe**, sin permisos administrativos, y su credencial se guardó por UI en `secrets/okta-trial-probe` (KV v2). Scope `vault.trial.probe`, política exclusiva del cliente; la regla no se creó porque la UI no ofrece Client Credentials.

La solicitud real al trial devuelve **HTTP 400 invalid_grant: The NHI Authentication Tokens SKU is not enabled**. No hay token emitido ni prueba OBO superada en esta organización. Okta debe habilitar el SKU antes de completar la regla y repetir la prueba.

Esta configuración de diagnóstico se creó por UI; no pertenece al estado Terraform de Integrator. El script mantiene DPoP y escribe solo evidencia no secreta en `docs/trial-probe-result.json`.

![Secreto del trial en Vault](docs/screenshots/21-vault-trial-secret.jpg)

In [ ]:
%%bash
set -euo pipefail
# El código de salida 1 refleja el rechazo OAuth observado.
.venv/bin/python scripts/trial_probe.py